<a href="https://colab.research.google.com/github/aashnikatari/eeg_classification/blob/eeg_sandbox/EEG_Data_Analysis_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
# Cell 2: Import Libraries and Configuration (now from config.py)
# =============================================================================

# Import all necessary libraries and configuration constants from config.py
!pip install mne

print("Libraries loaded successfully!")
print(f"MNE version: {mne.__version__}")
print(f"TensorFlow version: {tf.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 54.0 MB/s eta 0:00:00
Libraries loaded successfully!


NameError: name 'mne' is not defined

In [14]:
import os
from config import *

def copy_modules_to_gdrive_colab_path(destination_path):
    """
    Copies generated .py modules to the specified Google Drive Colab path.

    Parameters:
    -----------
    destination_path : str
        The Google Drive path where modules should be copied.
    """
    module_names = [
        'config.py',
        'eeg_loader.py',
        'eeg_preproc.py',
        'eeg_features.py',
        'eeg_linear.py',
        'eeg_lstm.py'
    ]

    os.makedirs(destination_path, exist_ok=True)

    print(f"Copying modules to: {destination_path}")
    for module_name in module_names:
        source_path = os.path.join('/content/', module_name)
        dest_file_path = os.path.join(destination_path, module_name)
        if os.path.exists(source_path):
            !cp "{source_path}" "{dest_file_path}"
            print(f"  Copied {module_name}")
        else:
            print(f"  Warning: {module_name} not found in /content/, skipping copy.")
    print("Module copying complete.")

def load_modules_from_gdrive_colab_path(source_path):
    """
    Copies generated .py modules from the specified Google Drive Colab path
    to the Colab runtime's /content/ directory.

    Parameters:
    -----------
    source_path : str
        The Google Drive path from where modules should be copied.
    """
    module_names = [
        'config.py',
        'eeg_loader.py',
        'eeg_preproc.py',
        'eeg_features.py',
        'eeg_linear.py',
        'eeg_lstm.py'
    ]

    # Ensure /content/ exists (it usually does)
    os.makedirs('/content/', exist_ok=True)

    print(f"Loading modules from: {source_path}")
    for module_name in module_names:
        source_file_path = os.path.join(source_path, module_name)
        dest_file_path = os.path.join('/content/', module_name)
        if os.path.exists(source_file_path):
            !cp "{source_file_path}" "{dest_file_path}"
            print(f"  Loaded {module_name}")
        else:
            print(f"  Warning: {module_name} not found in {source_path}, skipping load.")
    print("Module loading complete.")

print("Function `load_modules_from_gdrive_colab_path` defined.")

Function `load_modules_from_gdrive_colab_path` defined.


In [15]:
# =============================================================================
# Cell 3: Mount Google Drive & Setup Data Paths
# =============================================================================
# Upload your TDBRAIN subset to Google Drive before running this

from google.colab import drive
drive.mount('/content/gdrive')

# Set your data paths (modify these based on your folder structure)
TDBRAIN_PATH = '/content/gdrive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/'  # Updated path for 'Shared with me'
GDRIVE_PATH = '/content/gdrive/MyDrive/01_Research/01_Neuro_EEG/'
GDRIVE_OUTPUT_PATH = GDRIVE_PATH+'EEG_Analysis_Output/'
GDRIVE_COLAB_PATH  = GDRIVE_PATH+'Colab Notebooks'
OUTPUT_PATH = GDRIVE_OUTPUT_PATH # Explicitly define OUTPUT_PATH

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Expected folder structure:
# TDBRAIN_subset/
#   ├── participants.tsv        # Participant metadata with diagnosis
#   ├── sub-001/
#   │   └── eeg/
#   │       ├── sub-001_task-restEC_eeg.vhdr
#   │       ├── sub-001_task-restEC_eeg.vmrk
#   │       └── sub-001_task-restEC_eeg.eeg
#   ├── sub-002/
#   │   └── ...

print(f"TDBRAIN path: {TDBRAIN_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"GDrive Colab: {GDRIVE_COLAB_PATH}")

load_modules_from_gdrive_colab_path(GDRIVE_COLAB_PATH)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
TDBRAIN path: /content/gdrive/MyDrive/Shared with me/TDBRAIN_subset/TD-BRAIN-SAMPLE/
Output path: /content/gdrive/MyDrive/01_Research/01_Neuro_EEG/EEG_Analysis_Output/
GDrive Colab: /content/gdrive/MyDrive/01_Research/01_Neuro_EEG/Colab Notebooks
Loading modules from: /content/gdrive/MyDrive/01_Research/01_Neuro_EEG/Colab Notebooks
  Loaded config.py
  Loaded eeg_loader.py
  Loaded eeg_preproc.py
  Loaded eeg_features.py
  Loaded eeg_linear.py
  Loaded eeg_lstm.py
Module loading complete.


In [16]:
from eeg_loader import find_eeg_files, load_participants_metadata
from eeg_preproc import load_and_preprocess_eeg

print("EEG loading and preprocessing functions imported from respective modules!")

EEG loading and preprocessing functions imported from respective modules!


In [19]:
# =============================================================================
# Cell 4: Load Participant Metadata and Filter OCD Subjects
# =============================================================================
# The function `load_participants_metadata` is now part of eeg_loader.py
TDBRAIN_PATH = '/content/gdrive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/'  # Updated path for 'Shared with me'

# Load metadata
participants_df = load_participants_metadata(TDBRAIN_PATH)

# Filter OCD subjects (adjust column name based on your metadata)
if participants_df is not None:
    # Common column names in TDBRAIN: 'diagnosis', 'indication', 'group'
    diag_col = 'diagnosis' if 'diagnosis' in participants_df.columns else 'indication'
    # Ensure the column exists before filtering
    if diag_col in participants_df.columns:
        ocd_subjects = participants_df[participants_df[diag_col].str.contains(r'ADHD|SMC', case=False, na=False)]
    else:
        print(f"Warning: Neither 'diagnosis' nor 'indication' column found. Creating empty ocd_subjects DataFrame.")
        ocd_subjects = pd.DataFrame(columns=['participant_id', 'diagnosis'])
    print(f"\nOCD subjects found: {len(ocd_subjects)}")
    # Only try to print if there are columns to print
    if not ocd_subjects.empty and not ocd_subjects.columns.intersection(['participant_id', diag_col]).empty:
        print(ocd_subjects[['participant_id', diag_col]].head(10))
else:
    # If participants_df is None (should not happen with the dummy DataFrame logic now),
    # create an empty ocd_subjects DataFrame to prevent NameError
    ocd_subjects = pd.DataFrame(columns=['participant_id', 'diagnosis'])
    print("Warning: participants_df was None. Created empty ocd_subjects DataFrame.")

Total participants: 22

Columns: ['participant_id', 'indication', 'formal Dx', 'Dataset', 'Consent', 'sessSeason', 'sessTime', 'Responder', 'age', 'gender', 'sessID', 'nrSessions', 'EC', 'EO', 'neoFFI_q1', 'neoFFI_q2', 'neoFFI_q3', 'neoFFI_q4', 'neoFFI_q5', 'neoFFI_q6', 'neoFFI_q7', 'neoFFI_q8', 'neoFFI_q9', 'neoFFI_q10', 'neoFFI_q11', 'neoFFI_q12', 'neoFFI_q13', 'neoFFI_q14', 'neoFFI_q15', 'neoFFI_q16', 'neoFFI_q17', 'neoFFI_q18', 'neoFFI_q19', 'neoFFI_q20', 'neoFFI_q21', 'neoFFI_q22', 'neoFFI_q23', 'neoFFI_q24', 'neoFFI_q25', 'neoFFI_q26', 'neoFFI_q27', 'neoFFI_q28', 'neoFFI_q29', 'neoFFI_q30', 'neoFFI_q31', 'neoFFI_q32', 'neoFFI_q33', 'neoFFI_q34', 'neoFFI_q35', 'neoFFI_q36', 'neoFFI_q37', 'neoFFI_q38', 'neoFFI_q39', 'neoFFI_q40', 'neoFFI_q41', 'neoFFI_q42', 'neoFFI_q43', 'neoFFI_q44', 'neoFFI_q45', 'neoFFI_q46', 'neoFFI_q47', 'neoFFI_q48', 'neoFFI_q49', 'neoFFI_q50', 'neoFFI_q51', 'neoFFI_q52', 'neoFFI_q53', 'neoFFI_q54', 'neoFFI_q55', 'neoFFI_q56', 'neoFFI_q57', 'neoFFI_q58', 'neo

In [20]:
# =============================================================================
# Cell 7: Process All OCD Subjects and Build Feature Dataset
# =============================================================================
import pandas as pd
from eeg_features import create_epochs_and_features

def process_subjects(tdbrain_path, subject_ids, label, task='restEC'):
    """
    Process multiple subjects and extract features.

    Parameters:
    -----------
    tdbrain_path : str
        Path to TDBRAIN dataset
    subject_ids : list
        List of subject IDs to process
    label : str
        Label for these subjects (e.g., 'OCD', 'Anxiety', 'Control')
    task : str
        Task name for EEG files

    Returns:
    --------
    all_features : pd.DataFrame
        DataFrame with all extracted features and labels
    """
    all_features = []

    for subj_id in subject_ids:
        print(f"Processing {subj_id}...")
        subject_path = os.path.join(tdbrain_path, subj_id)

        # Find EEG file
        vhdr_file = find_eeg_files(subject_path, task)
        if vhdr_file is None:
            print(f"  No EEG file found for {subj_id}")
            continue

        # Load and preprocess
        raw = load_and_preprocess_eeg(vhdr_file, TARGET_SFREQ, COMMON_CHANNELS)
        if raw is None:
            continue

        # Debugging: Print raw data duration
        print(f"  Raw data duration for {subj_id}: {raw.times[-1]:.2f} seconds")

        # Extract features from epochs
        epoch_features = create_epochs_and_features(raw, EPOCH_DURATION, FREQ_BANDS)

        if not epoch_features:
            print(f"  No epochs created for {subj_id}. Data might be too short for {EPOCH_DURATION}-second epochs.")

        # Add subject info and label to each epoch
        for i, features in enumerate(epoch_features):
            features['subject_id'] = subj_id
            features['epoch'] = i
            features['label'] = label
            all_features.append(features)

        print(f"  Extracted {len(epoch_features)} epochs")

    return pd.DataFrame(all_features)

TDBRAIN_PATH = '/content/gdrive/MyDrive/TDBRAIN_subset/TD-BRAIN-SAMPLE/'  # Updated path for 'Shared with me'

# Example usage (uncomment when you have the data):
ocd_subject_ids = ocd_subjects['participant_id'].tolist()[:30]  # Limit to 30 subjects
ocd_features = process_subjects(TDBRAIN_PATH, ocd_subject_ids, 'ADHD')
print(f"Total OCD features: {len(ocd_features)}")

print("Subject processing function defined!")
print("Uncomment the example usage above when your data is ready.")

Processing sub-87966293...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  Downsampled: 500.0 Hz -> 256 Hz
  Raw data duration for sub-87966293: 120.64 seconds
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
  Extracted 12 epochs
Processing sub-87966293...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
  Downsampled: 500.0 Hz -> 256 Hz
  Raw data duration for sub-87966293: 120.64 seconds
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective window size : 8.000 (s)
Effective wi

In [21]:
# =============================================================================
# Cell 8: ML Model Training - Classical Models (SVM, Random Forest)
# =============================================================================
from eeg_linear import prepare_data_for_ml, train_classical_models

# Example usage (uncomment when you have processed features):
X, y, feature_names, label_encoder = prepare_data_for_ml(ocd_features)
results, scaler = train_classical_models(X, y)

print("Classical ML training functions defined!")

eeg_linear.py recreated successfully!

Training Random Forest...
  Accuracy: 1.0000 (+/- 0.0000)
Classical ML training functions defined!


In [22]:
# =============================================================================
# Cell 9: Deep Learning Model - LSTM
# =============================================================================
from eeg_lstm import prepare_data_for_lstm, build_lstm_model, train_lstm_model

# Example usage (uncomment when you have features):
model, history, lstm_scaler, test_data = train_lstm_model(X, y)

print("LSTM training functions defined!")

eeg_lstm.py recreated successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 64)          │        45,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,785 (229.63 KB)

 Trainable params: 58,785 (229.63 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 356ms/step - accuracy: 0.4561 - loss: 0.6951 - val_accuracy: 0.8182 - val_loss: 0.6813
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.8748 - loss: 0.6752 - val_accuracy: 0.9091 - val_loss: 0.6653
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.8944 - loss: 0.6583 - val_accuracy: 0.9091 - val_loss: 0.6468
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 0.6382 - val_accuracy: 0.9091 - val_loss: 0.6262
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 1.0000 - loss: 0.6126 - val_accuracy: 0.9545 - val_loss: 0.6021
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 1.0000 - loss: 0.5907 - val_accuracy: 0.9545 - val_loss: 0.5741
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 1.0000 - loss: 0.5525 - val_accuracy: 0.9545 - val_loss: 0.5420
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 1.0000 - loss: 0.5248 - val_accuracy: 1.0000 - val_loss: 0

In [1]:
# =============================================================================
# Cell 10: Visualization and Save Results
# =============================================================================

def plot_training_history(history):
    """
    Plot training and validation accuracy/loss curves.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'training_history.png'), dpi=150)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes):
    """
    Plot confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'confusion_matrix.png'), dpi=150)
    plt.show()

def compare_models(results):
    """
    Compare performance of different models.
    """
    models = list(results.keys())
    accuracies = [results[m]['mean_accuracy'] for m in models]
    stds = [results[m]['std_accuracy'] for m in models]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(models, accuracies, yerr=stds, capsize=5, color='steelblue', alpha=0.8)
    plt.ylabel('Accuracy')
    plt.title('Model Comparison - OCD vs Anxiety Classification')
    plt.ylim(0, 1)

    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'model_comparison.png'), dpi=150)
    plt.show()

def save_results(features_df, results, output_path):
    """
    Save processed features and results to files.
    """
    # Save features
    features_df.to_csv(os.path.join(output_path, 'extracted_features.csv'), index=False)
    print(f"Features saved to {output_path}/extracted_features.csv")

    # Save results summary
    results_summary = pd.DataFrame([
        {'Model': name, 'Mean_Accuracy': r['mean_accuracy'], 'Std_Accuracy': r['std_accuracy']}
        for name, r in results.items()
    ])
    results_summary.to_csv(os.path.join(output_path, 'results_summary.csv'), index=False)
    print(f"Results saved to {output_path}/results_summary.csv")

print("Visualization and saving functions defined!")
print("\n" + "="*70)
print("NOTEBOOK SETUP COMPLETE!")
print("="*70)
print("\nNext steps:")
print("1. Upload your TDBRAIN subset to Google Drive")
print("2. Update TDBRAIN_PATH in Cell 3")
print("3. Run cells in order to process data and train models")
print("4. Adjust parameters (epochs, batch size) based on your compute resources")

Visualization and saving functions defined!

NOTEBOOK SETUP COMPLETE!

Next steps:
1. Upload your TDBRAIN subset to Google Drive
2. Update TDBRAIN_PATH in Cell 3
3. Run cells in order to process data and train models
4. Adjust parameters (epochs, batch size) based on your compute resources
